# Stage 1 — long-format item data, cleaning, subsampling (reading = primary, mathematics = replication)
Input: Drive/PISA2025_extract (from the extraction notebook). Runtime → Run all. Output: Drive/PISA2025_stage1.

In [ ]:
import os, glob, gc, numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
IN='/content/drive/MyDrive/PISA2025_extract'; OUT='/content/drive/MyDrive/PISA2025_stage1'; os.makedirs(OUT,exist_ok=True)
CNTS=['ARM','KAZ','MDA','LVA','LTU','EST','JPN','ARE','SGP']
DOMS={'read':'CR','math':'CM'}; NSUB=3000; SEED=20251025; rng=np.random.default_rng(SEED)
STUV=['CNT','CNTSCHID','CNTSTUID','ST004D01T','ESCS','AIUSESCH','EFFORT1','EFFORT2','W_FSTUWT','ST438Q01DA','ST438Q02DA','ST438Q03DA','ST438Q04DA','PV1READ','PV1MATH']

def to_num(s):
    v=pd.to_numeric(s,errors='coerce'); return v.where(v<9e7)        # 99999997/8/9 -> NaN

desc=[]; items_all=[]
for c in CNTS:
    stu=pd.read_parquet(f'{IN}/STU_{c}.parquet'); stu=stu[[v for v in STUV if v in stu]]
    ids=stu.CNTSTUID.values  # full samples (subsampling moved to modelling stage)
    stu=stu[stu.CNTSTUID.isin(ids)]
    cog=pd.read_parquet(f'{IN}/COG_{c}.parquet'); cog=cog[cog.CNTSTUID.isin(ids)]
    prc=pd.read_parquet(f'{IN}/PROC_{c}.parquet'); prc=prc[prc.CNTSTUID.isin(ids)]
    for dom,pre in DOMS.items():
        sc=[x for x in cog if x.startswith(pre) and x.endswith('S') and 'Q' in x]
        L=[]
        for s in sc:
            it=s[:-1]
            if it+'TT' not in prc: continue
            raw=cog[['CNTSTUID',s]].rename(columns={s:'raw'}); raw['raw']=raw['raw'].astype(str).str.strip()
            p=prc[['CNTSTUID']].copy(); p['TT']=prc[it+'TT']; p['A']=prc[it+'A'] if it+'A' in prc else np.nan; p['F']=prc[it+'F'] if it+'F' in prc else np.nan
            for k in ['TT','A','F']: p[k]=to_num(p[k])
            m=raw.merge(p,on='CNTSTUID',how='left'); m=m[m.raw.isin(['0','1','2','r'])]
            if m.empty: continue
            m['item']=it; L.append(m)
        if not L: continue
        d=pd.concat(L,ignore_index=True)
        poly=set(d.loc[d.raw=='2','item']); d['dich']=~d.item.isin(poly)
        d['not_reached']=d.raw.eq('r'); d['x']=pd.to_numeric(d.raw.where(d.raw.isin(['0','1','2'])),errors='coerce')
        d['t']=d.TT/1000.0; d['f']=d.F/1000.0                                   # ms -> s
        d['valid_t']=d.t.gt(0)&(d.f.isna()|d.f.le(d.t))
        q=d[d.valid_t&~d.not_reached].groupby('item').t.agg(mean_t='mean',q995=lambda s:s.quantile(.995)).reset_index()
        d=d.merge(q,on='item',how='left')
        d['nt10']=np.minimum(0.10*d.mean_t,10.0)                               # Wise & Ma (2012) NT10 threshold
        d['rapid']=d.valid_t&d.t.lt(d.nt10)
        d['trim']=d.t.gt(d.q995)
        d['CNT']=c; d['domain']=dom
        d=d.merge(stu,on=['CNTSTUID'],how='left',suffixes=('','_s'))
        d.drop(columns=[k for k in d if k.endswith('_s')]+['TT','F','raw','mean_t','q995'],inplace=True)
        d.to_parquet(f'{OUT}/long_{dom}_{c}.parquet',index=False)
        ok=d[d.valid_t&~d.trim&~d.not_reached&d.dich]
        desc.append({'CNT':c,'domain':dom,'students':ok.CNTSTUID.nunique(),'items':ok.item.nunique(),'poly_items_excl':len(poly),
            'resp_total':len(ok),'items_per_stu':round(ok.groupby('CNTSTUID').size().mean(),2),'acc':round(ok.x.mean(),3),
            'med_t_s':round(ok.t.median(),1),'med_f_s':round(ok.f.median(),1),'med_A':ok.A.median(),
            'rapid_pct':round(100*d.loc[d.valid_t,'rapid'].mean(),2),'not_reached_pct':round(100*d.not_reached.mean(),2),
            'f_missing_pct':round(100*d.f.isna().mean(),2),'invalid_t_pct':round(100*(~d.valid_t&~d.not_reached).mean(),2),'ai_cov':round(stu.AIUSESCH.notna().mean(),3),
            'ai_cov_in_resp':round(ok.drop_duplicates('CNTSTUID').AIUSESCH.notna().mean(),3)})
        items_all.append(ok.groupby('item').agg(n=('x','size'),p=('x','mean'),med_t=('t','median')).assign(CNT=c,domain=dom).reset_index())
        del d,ok; gc.collect()
    del stu,cog,prc; gc.collect(); print(c,'done',flush=True)
D=pd.DataFrame(desc); D.to_csv(f'{OUT}/stage1_desc.csv',index=False)
I=pd.concat(items_all); I.to_csv(f'{OUT}/stage1_items.csv',index=False)
cov=I.groupby(['domain','item']).agg(k=('CNT','nunique'),nmin=('n','min')).reset_index()
print(D.to_string()); print('items in all 9 countries with n>=200:', cov[(cov.k==9)&(cov.nmin>=200)].groupby('domain').size().to_dict()); print('STAGE1 DONE')
